# Simple HandWriting OCR

This simple handwriting will involve custom CNN model by using Tensorflow.
<br>I will call it MinimNet and it can be found in model.py

## 1. Import All and Prepare the Dataset
Before going on, we could download the dataset from the [link Sueiras](https://s3-eu-west-1.amazonaws.com/handwriting-curated-database/curated.tar.gz) have given.<br>
And just unzip the file in the same directory of this notebook.

In [1]:
from model2 import MinimNet
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from imutils import resize, grab_contours
from matplotlib import pyplot as plt
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from PIL import Image
import numpy as np
import cv2
import os
import bitstring
import glob

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

I0000 00:00:1782676550.260671   78414 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


import all needed libraries and functions needed

And before we are going to prepare the dataset, we would want to set our parameters first

1. Dataset path,
2. Image size,
3. Batch size,
4. Number of epochs,
5. Initial Learning Rate (LR)
6. Ratio of train/test

In [2]:
notebook_path = os.path.abspath('Simple-OCR_runpod.ipynb')
dataset_path = os.path.dirname(notebook_path)+'/curated'
etl1_dir = os.path.join(os.path.dirname(notebook_path), 'ETL/ETL1')
etl1_paths = [
    os.path.join(etl1_dir, f) for f in os.listdir(etl1_dir) if f.startswith('ETL1C_')
] if os.path.exists(etl1_dir) else []

image_size = (32, 32)
batch_size = 512
epoch = 100
init_lr = 1e-1
test_ratio = .25

After putting all the needed parameters, we then will try to load the images from the path we defined

We will need both the image and label, which the label is the folder name (ASCII Decimal)

Thus we need somekind of label to make it more uniformed.<br> Then we will make a classLabels that consist of needed images to train.

After that we will load all of the images that we need along with the labels<br>
And change the labels into a one-hot label.

In [9]:
import os
import random
import numpy as np
import bitstring
import pandas as pd
from PIL import Image, ImageDraw
from scipy.ndimage import maximum_filter

# =====================================================================
# 0. KHỞI TẠO CẤU HÌNH VÀ HÀM BỔ TRỢ (HELPER FUNCTIONS)
# =====================================================================
classLabels = "1234567890ABCDEFGHIJKLMNOPQRSTUVWXYZ[]"
classLabels = [i for i in classLabels]

img = []
lbl = []

# Kiểm tra an toàn biến môi trường
if 'image_size' not in locals() and 'image_size' not in globals():
    image_size = (32, 32)
if 'dataset_path' not in locals() and 'dataset_path' not in globals():
    dataset_path = "./dataset" 

def make_it_bold(img_array):
    """Làm đậm nét chữ bằng thuật toán giãn nở."""
    return maximum_filter(img_array, size=(3, 3))

def make_it_broken(img_array, num_holes=6, hole_size=2):
    """Giả lập nét đứt/mực mờ bằng cách đục lỗ đen ngẫu nhiên."""
    broken_img = np.copy(img_array)
    h, w = broken_img.shape
    for _ in range(num_holes):
        y = random.randint(0, max(0, h - hole_size - 1))
        x = random.randint(0, max(0, w - hole_size - 1))
        broken_img[y:y+hole_size, x:x+hole_size] = 0.0
    return broken_img

def resize_with_pad(img_obj, target_size=(32, 32)):
    """Resize giữ nguyên tỷ lệ gốc, viền đen (Chống bóp méo chữ 1 và I)."""
    w, h = img_obj.size
    ratio = min(target_size[0] / w, target_size[1] / h)
    new_w = max(1, int(w * ratio))
    new_h = max(1, int(h * ratio))
    
    img_resized = img_obj.resize((new_w, new_h), Image.LANCZOS)
    new_img = Image.new('L', target_size, color=0)
    new_img.paste(img_resized, ((target_size[0] - new_w) // 2, (target_size[1] - new_h) // 2))
    return new_img

def generate_realistic_brackets(char, num_samples=1000, img_size=(32, 32)):
    """Sinh dữ liệu dấu '[' hoặc ']' giả lập góc méo, nét ko đều thực tế."""
    synthetic_imgs = []
    W, H = img_size
    
    for _ in range(num_samples):
        canvas = Image.new('L', (64, 64), color=0)
        draw = ImageDraw.Draw(canvas)
        
        cx, cy = 32, 32
        h_half = random.randint(14, 20)  
        w_len = random.randint(12, 18)   # NÉT NGANG DÀI HƠN để tránh giống chữ I
        
        # NÉT NGANG DÀY HƠN NÉT DỌC để bảo vệ cấu trúc khi bị đứt nét
        thick_vertical = random.randint(2, 4)   
        thick_horizontal = random.randint(3, 5) 
        
        skew_top = random.randint(-3, 3)    
        skew_bottom = random.randint(-3, 3) 
        skew_vertical = random.randint(-2, 2) 
        
        if char == '[':
            v_top = (cx - w_len//2 + skew_vertical, cy - h_half)
            v_bottom = (cx - w_len//2 - skew_vertical, cy + h_half)
            h_top_end = (v_top[0] + w_len, v_top[1] + skew_top)
            h_bottom_end = (v_bottom[0] + w_len, v_bottom[1] + skew_bottom)
            
            draw.line([v_top, v_bottom], fill=255, width=thick_vertical)
            draw.line([v_top, h_top_end], fill=255, width=thick_horizontal)
            draw.line([v_bottom, h_bottom_end], fill=255, width=thick_horizontal)
        else: 
            v_top = (cx + w_len//2 + skew_vertical, cy - h_half)
            v_bottom = (cx + w_len//2 - skew_vertical, cy + h_half)
            h_top_end = (v_top[0] - w_len, v_top[1] + skew_top)
            h_bottom_end = (v_bottom[0] - w_len, v_bottom[1] + skew_bottom)
            
            draw.line([v_top, v_bottom], fill=255, width=thick_vertical)
            draw.line([v_top, h_top_end], fill=255, width=thick_horizontal)
            draw.line([v_bottom, h_bottom_end], fill=255, width=thick_horizontal)
            
        angle = random.uniform(-6, 6)    # Góc đứng thẳng hơn
        canvas = canvas.rotate(angle, resample=Image.BICUBIC)
        
        shift_x = random.randint(-3, 3)
        shift_y = random.randint(-3, 3)
        left = (64 - W) // 2 + shift_x
        top = (64 - H) // 2 + shift_y
        
        canvas = canvas.crop((left, top, left + W, top + H))
        img_array = np.array(canvas) * (1. / 255.)
        synthetic_imgs.append(img_array)
        
    return synthetic_imgs

def load_etl1_data(file_paths, class_labels, img_size=(32, 32)):
    """Đọc dữ liệu từ danh sách file ETL và sinh thêm phiên bản nét đậm, đứt gãy."""
    etl_img = []
    etl_lbl = []
    
    for file_path in file_paths:
        if not os.path.exists(file_path):
            continue
            
        print(f"Đang đọc ETL file: {file_path}...")
        file_size = os.path.getsize(file_path)
        record_size = 2052
        num_records = file_size // record_size
        
        with open(file_path, "rb") as f:
            for i in range(num_records):
                _bytes = f.read(record_size)
                if len(_bytes) < record_size: break
                    
                f_bit = bitstring.ConstBitStream(bytes=_bytes)
                r = f_bit.readlist('uint:16,bytes:2,uint:16,hex:8,hex:8,4*uint:8,uint:32,4*uint:16,4*uint:8,pad:32,bytes:2016,pad:32')
                char_code = r[1][0]
                img_bytes = r[18]
                
                if 32 <= char_code <= 126:
                    char = chr(char_code)
                    if char in class_labels:
                        lbl_idx = class_labels.index(char)
                        pixels = []
                        for b in img_bytes:
                            pixels.extend([(b >> 4) & 0x0F, b & 0x0F])
                            
                        img_array = 255 - (np.array(pixels, dtype=np.uint8) * 17)
                        img_array = img_array.reshape(63, 64)
                        img_obj = Image.fromarray(img_array)
                        
                        # Data Augmentation cho ETL1
                        def add_augmented_samples(base_img_obj):
                            img_resized = resize_with_pad(base_img_obj, img_size)
                            img_norm = np.array(img_resized) * (1. / 255.)
                            
                            # Thêm trục channel axis=-1 để tạo mảng 3D chuẩn (32, 32, 1)
                            etl_img.append(np.expand_dims(img_norm, axis=-1))
                            etl_lbl.append(lbl_idx)
                            
                            etl_img.append(np.expand_dims(make_it_bold(img_norm), axis=-1))
                            etl_lbl.append(lbl_idx)
                            
                            etl_img.append(np.expand_dims(make_it_broken(img_norm), axis=-1))
                            etl_lbl.append(lbl_idx)

                        # Mẫu gốc chuẩn
                        add_augmented_samples(img_obj)
                        
                        # Xoay thêm ảnh nếu là các ký tự dễ nhầm lẫn
                        if char in ['1', '7', 'I']:
                            angle = random.uniform(-10, 10)
                            img_rot = img_obj.rotate(angle, resample=Image.BICUBIC, fillcolor=0)
                            add_augmented_samples(img_rot)
                                        
    return etl_img, etl_lbl

# =====================================================================
# TÍCH HỢP DỮ LIỆU TỪ TRDG (Synthetic Data)
# =====================================================================
def load_synthetic_data(folder, classLabels):
    print("DEBUG: [1] Đang quét thư mục synthetic...")
    file_paths = glob.glob(os.path.join(folder, "*.jpg"))
    print(f"DEBUG: [2] Số file tìm thấy trong thư mục: {len(file_paths)}")
    
    # Chuyển classLabels sang kiểu chuỗi (chuẩn hóa xóa khoảng trắng) để so sánh chính xác
    valid_labels = [str(c).strip() for c in classLabels]
    
    s_imgs = []
    s_lbls = []
    
    # Cấu hình kích thước chuẩn theo tập dữ liệu hiện tại của bạn
    IMG_WIDTH = 32  
    IMG_HEIGHT = 32
    
    for i, file_path in enumerate(file_paths):
        filename = os.path.basename(file_path)
        
        # Tách nhãn nằm trước dấu gạch dưới '_'
        label_str = filename.split('_')[0].strip()
        
        if label_str in valid_labels:
            # Đọc ảnh dạng ảnh xám (2D)
            img = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
            
            if img is not None:
                # Ép kích thước về 32x32 để tránh lỗi không đồng nhất hình dạng (inhomogeneous shape)
                img_resized = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
                
                s_imgs.append(img_resized)
                s_lbls.append(int(label_str)) # Lưu nhãn dưới dạng số nguyên gốc
            else:
                if i < 5:
                    print(f"⚠️ DEBUG: Không thể đọc tệp tin: {filename}")
                    
    print(f"DEBUG: [3] Kết thúc load. Tổng số ảnh hợp lệ trích xuất được: {len(s_imgs)}")
    return np.array(s_imgs), np.array(s_lbls)

# ĐƯỜNG DẪN MỚI SAU KHI GIẢI NÉN
notebook_path = os.getcwd() # Hoặc sử dụng đường dẫn tuyệt đối của notebook
hasy_base_dir = os.path.join(notebook_path, 'HASYV2')
hasy_csv_path = os.path.join(hasy_base_dir, 'hasy-data-labels.csv')

# Các hàm helper (make_it_bold, make_it_broken, resize_with_pad, etc.)
# Vẫn giữ nguyên như cũ, chỉ cập nhật logic load dữ liệu bên dưới...

def load_hasy_data(csv_path, class_labels):
    """Đọc dữ liệu từ Dataset HASY với cấu trúc thư mục mới."""
    hasy_img = []
    hasy_lbl = []
    if not os.path.exists(csv_path):
        print(f"⚠️ Không tìm thấy file HASY CSV tại: {csv_path}")
        return hasy_img, hasy_lbl
        
    print(f"Bắt đầu đọc dữ liệu HASY từ CSV: {csv_path}...")
    df = pd.read_csv(csv_path)
    # Ảnh nằm trong HASYV2/hasy-data/
    # File csv nằm trong HASYV2/
    # Cột 'path' trong csv thường là 'hasy-data/v2-XXXX.png'
    # Nên đường dẫn đầy đủ là: hasy_base_dir + row['path']
    
    df_filtered = df[df['latex'].isin(class_labels)]
    
    for _, row in df_filtered.iterrows():
        img_path = os.path.join(hasy_base_dir, row['path'])
        char = row['latex']
        lbl_idx = class_labels.index(char)
        
        if not os.path.exists(img_path): continue
            
        try:
            image_obj = Image.open(img_path).convert('L')
            # Đảo màu (Invert) cho HASY vì nó là đen trên trắng
            image_obj = ImageOps.invert(image_obj)
            
            # Augmentation (Gốc + Đậm + Đứt)
            process_and_augment(image_obj, char, lbl_idx, hasy_img, hasy_lbl)
        except Exception:
            pass
            
    return hasy_img, hasy_lbl
    
# =====================================================================
# 1. XỬ LÝ CURATED DATASET
# =====================================================================
if os.path.exists(dataset_path):
    print("Bắt đầu xử lý Curated Dataset...")
    for folder in os.listdir(dataset_path):
        folder_path = os.path.join(dataset_path, folder)
        if os.path.isdir(folder_path):
            try:
                folder_char = chr(int(folder))
            except ValueError:
                continue
                
            if folder_char in classLabels:
                lbl_idx = classLabels.index(folder_char)
                for file in os.listdir(folder_path):
                    if file.startswith('.'): continue
                    file_path = os.path.join(folder_path, file)
                    
                    try:
                        image_obj = Image.open(file_path)
                        
                        def add_curated_samples(base_obj):
                            img_resized = resize_with_pad(base_obj, image_size)
                            img_norm = np.array(img_resized) * (1. / 255.)
                            
                            img.append(np.expand_dims(img_norm, axis=-1))
                            lbl.append(lbl_idx)
                            
                            img.append(np.expand_dims(make_it_bold(img_norm), axis=-1))
                            lbl.append(lbl_idx)
                            
                            img.append(np.expand_dims(make_it_broken(img_norm), axis=-1))
                            lbl.append(lbl_idx)

                        add_curated_samples(image_obj)
                        
                        if folder_char in ['1', '7', 'I']:
                            angle = random.uniform(-10, 10)
                            img_rot = image_obj.rotate(angle, resample=Image.BICUBIC, fillcolor=0)
                            add_curated_samples(img_rot)
                            
                    except Exception as e:
                        pass

img = np.array(img, dtype='float32') if len(img) > 0 else np.empty((0, *image_size, 1), dtype='float32')
lbl = np.array(lbl) if len(lbl) > 0 else np.empty((0,), dtype='int64')
print(f"✅ Xong Curated Dataset: {len(img)} ảnh (đã x3 với Đậm và Đứt nét).")

# =====================================================================
# 2. XỬ LÝ DỮ LIỆU TỪ TẬP HASY
# =====================================================================
if 'hasy_csv_path' in locals() or 'hasy_csv_path' in globals():
    hasy_img, hasy_lbl = load_hasy_data(hasy_csv_path, classLabels)
    if len(hasy_img) > 0:
        img = np.concatenate([img, np.array(hasy_img, dtype='float32')], axis=0)
        lbl = np.concatenate([lbl, np.array(hasy_lbl)], axis=0)
        print(f"✅ Đã thêm {len(hasy_img)} ảnh từ HASY Dataset.")


# =====================================================================
# 2. TỰ SINH DỮ LIỆU NGOẶC THỰC TẾ CHO '[' VÀ ']'
# =====================================================================
# print("Bắt đầu tự sinh dữ liệu cho ký tự '[' và ']'...")
# bracket_chars = ['[', ']']
# num_samples_per_char = 1500  

# bracket_imgs = []
# bracket_lbls = []

# for char in bracket_chars:
#     lbl_idx = classLabels.index(char)
#     syn_imgs = generate_realistic_brackets(char, num_samples=num_samples_per_char, img_size=image_size)
    
#     for img_arr in syn_imgs:
#         bracket_imgs.append(np.expand_dims(img_arr, axis=-1))
#         bracket_lbls.append(lbl_idx)
        
#         bracket_imgs.append(np.expand_dims(make_it_bold(img_arr), axis=-1))
#         bracket_lbls.append(lbl_idx)
        
#         bracket_imgs.append(np.expand_dims(make_it_broken(img_arr), axis=-1))
#         bracket_lbls.append(lbl_idx)

# if len(bracket_imgs) > 0:
#     bracket_imgs = np.array(bracket_imgs, dtype='float32')
#     bracket_lbls = np.array(bracket_lbls)
    
#     if len(img) > 0:
#         img = np.concatenate([img, bracket_imgs], axis=0)
#         lbl = np.concatenate([lbl, bracket_lbls], axis=0)
#     else:
#         img = bracket_imgs
#         lbl = bracket_lbls
#     print(f"✅ Đã thêm {len(bracket_imgs)} ảnh ngoặc (gồm cả Đậm và Đứt nét).")

# =====================================================================
# 1. CÁC BƯỚC LOAD DỮ LIỆU
# =====================================================================
# ... (Code load Curated Dataset cũ)

# --- CẬP NHẬT: Load và Hòa trộn vào mảng chính ---
# syn_folder = "./synthetic_data" # Đường dẫn thư mục ảnh trdg của bạn
# if os.path.exists(syn_folder):
#     # Gọi hàm load dữ liệu chuẩn hóa
#     s_imgs, s_lbls = load_synthetic_data(syn_folder, classLabels)
    
#     if len(s_imgs) > 0:
#         # Kiểm tra mảng cũ 'img' có tồn tại và hợp lệ không
#         img_exists = 'img' in locals() and img is not None and len(img) > 0
        
#         if img_exists:
#             # Đồng bộ số chiều: Nếu mảng cũ là 4D (N, H, W, C) và mảng mới là 3D (N, H, W)
#             if len(img.shape) == 4 and len(s_imgs.shape) == 3:
#                 s_imgs = np.expand_dims(s_imgs, axis=-1)
#                 print(f"🔄 Đã đồng bộ số chiều mảng mới về 4D: {s_imgs.shape}")
                
#             # Tiến hành hòa trộn dữ liệu một cách an toàn
#             img = np.concatenate([img, s_imgs], axis=0)
#             lbl = np.concatenate([lbl, s_lbls], axis=0)
#         else:
#             # Nếu mảng cũ chưa tồn tại, khởi tạo mảng từ tập dữ liệu mới
#             if len(s_imgs.shape) == 3:
#                 s_imgs = np.expand_dims(s_imgs, axis=-1)
#             img, lbl = s_imgs, s_lbls
            
#         print(f"✅ Đã hòa trộn thành công {len(s_imgs)} ảnh từ TRDG vào tập train.")
#         print(f"📊 Tổng số lượng ảnh sau khi gộp: {len(img)} | Shape hiện tại: {img.shape}")
        
#         # --- KHUYÊN DÙNG: Xáo trộn (Shuffle) lại dữ liệu sau khi gộp ---
#         indices = np.arange(len(img))
#         np.random.shuffle(indices)
#         img = img[indices]
#         lbl = lbl[indices]
#         print("🔀 Đã hoàn thành xáo trộn dữ liệu (Shuffle) toàn tập.")
        
# else:
#     print(f"⚠️ Không tìm thấy thư mục {syn_folder}. Vui lòng kiểm tra lại đường dẫn.")

# LOAD HASY DỮ LIỆU (MỚI)
if os.path.exists(hasy_csv_path):
    h_img, h_lbl = load_hasy_data(hasy_csv_path, classLabels)
    if len(h_img) > 0:
        if len(img) == 0:
            img = np.array(h_img, dtype='float32')
            lbl = np.array(h_lbl)
        else:
            img = np.concatenate([img, np.array(h_img, dtype='float32')], axis=0)
            lbl = np.concatenate([lbl, np.array(h_lbl)], axis=0)
        print(f"✅ Đã thêm {len(h_img)} ảnh từ HASY Dataset.")
        
# =====================================================================
# 3. TỰ ĐỘNG QUÉT VÀ GỘP THÊM TOÀN BỘ DỮ LIỆU TỪ THƯ MỤC ETL/
# =====================================================================
# =====================================================================
# 3. TỰ ĐỘNG QUÉT VÀ GỘP THÊM TOÀN BỘ DỮ LIỆU TỪ THƯ MỤC ETL/
# =====================================================================
etl_base_dir = "./ETL"  # Đường dẫn tới thư mục ETL tổng của bạn

if os.path.exists(etl_base_dir):
    print(f"\n🔍 Đang quét tự động các tệp tin dữ liệu trong: {etl_base_dir}...")
    
    # Tìm tất cả các file bên trong thư mục ETL và các thư mục con của nó
    all_etl_files = glob.glob(os.path.join(etl_base_dir, "**/*"), recursive=True)
    
    # Lọc bỏ các thư mục và chỉ giữ lại các file dữ liệu ETL thực sự
    etl_files = [f for f in all_etl_files if os.path.isfile(f) and not os.path.basename(f).startswith('.')]
    
    if len(etl_files) > 0:
        print(f"📊 Tìm thấy tổng cộng {len(etl_files)} tệp dữ liệu ETL để xử lý.")
        
        # Gọi hàm load dữ liệu từ danh sách file vừa quét được
        etl_img, etl_lbl = load_etl1_data(etl_files, classLabels, image_size)
        
        if len(etl_img) > 0:
            etl_img_arr = np.array(etl_img, dtype='float32')
            etl_lbl_arr = np.array(etl_lbl, dtype='int64')
            
            if len(img) == 0:
                img = etl_img_arr
                lbl = etl_lbl_arr
            else:
                img = np.concatenate([img, etl_img_arr], axis=0)
                lbl = np.concatenate([lbl, etl_lbl_arr], axis=0)
                
            print(f"✅ Đã thêm {len(etl_img)} ảnh từ toàn bộ thư mục ETL.")
    else:
        print(f"⚠️ Thư mục {etl_base_dir} rỗng hoặc không chứa tệp hợp lệ.")
else:
    print(f"⚠️ Không tìm thấy thư mục cấu hình {etl_base_dir}.")


# =====================================================================
# 3.1 Xáo trộn lần cuối
# =====================================================================
indices = np.arange(len(img))
np.random.shuffle(indices)
img = img[indices]
lbl = lbl[indices]

# =====================================================================
# 4. KẾT QUẢ ĐẦU RA TỔNG HỢP
# =====================================================================
print("\n--- THÔNG TIN TẬP DỮ LIỆU CUỐI CÙNG ---")
print(f"✅ Tổng số lượng ảnh: {len(img)}")
print(f"✅ Shape tổng thể của tập dữ liệu: {img.shape}")
print(f"✅ Shape của tập nhãn: {lbl.shape}")

Bắt đầu xử lý Curated Dataset...
✅ Xong Curated Dataset: 89226 ảnh (đã x3 với Đậm và Đứt nét).
Bắt đầu đọc dữ liệu HASY từ CSV: /workspace/HASYV2/hasy-data-labels.csv...
Bắt đầu đọc dữ liệu HASY từ CSV: /workspace/HASYV2/hasy-data-labels.csv...

🔍 Đang quét tự động các tệp tin dữ liệu trong: ./ETL...
📊 Tìm thấy tổng cộng 16 tệp dữ liệu ETL để xử lý.
Đang đọc ETL file: ./ETL/ETL1/ETL1INFO...
Đang đọc ETL file: ./ETL/ETL1/ETL1C_13...
Đang đọc ETL file: ./ETL/ETL1/ETL1C_12...
Đang đọc ETL file: ./ETL/ETL1/ETL1C_11...
Đang đọc ETL file: ./ETL/ETL1/ETL1C_10...
Đang đọc ETL file: ./ETL/ETL1/ETL1C_09...
Đang đọc ETL file: ./ETL/ETL1/ETL1C_08...
Đang đọc ETL file: ./ETL/ETL1/ETL1C_07...
Đang đọc ETL file: ./ETL/ETL1/ETL1C_06...
Đang đọc ETL file: ./ETL/ETL1/ETL1C_05...
Đang đọc ETL file: ./ETL/ETL1/ETL1C_04...
Đang đọc ETL file: ./ETL/ETL1/ETL1C_02...
Đang đọc ETL file: ./ETL/ETL1/ETL1C_01...
Đang đọc ETL file: ./ETL/ETL1/ETL1C_03...
Đang đọc ETL file: ./ETL/ETL3/ETL3C_2...
Đang đọc ETL file: 

In [10]:
#create a one-hot label
le = LabelBinarizer()
lbl = le.fit_transform(lbl)

#count the total number of images per class
classTotals = lbl.sum(axis=0)
classWeight = {}

#create a custom weight for every class
for i in range(len(classTotals)):
    classWeight[i] = classTotals.max() / classTotals[i]
    
#split the class to train and test using scikit train_test_split
(trainImg, testImg, trainLbl, testLbl) = train_test_split(img, lbl,
                                                         test_size = test_ratio,
                                                         stratify = lbl,
                                                         random_state = 42)

After we done in splitting the dataset, we will proceed to the next step.
<br><br>

## 2. Create the Model to train

As I have said before, we will try to build a model.<br>
This model will be consisting of some layers such as:
1. Bottleneck Layer,
2. Inverted Bottleneck Layer,
3. Squeeze Global Layer,
4. SE Module.

The model could be checked in the Model.py or in the summary below.

In [11]:
model = MinimNet((image_size[0], image_size[1], 1), len(classTotals))
model.summary()

from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.optimizers import SGD

# 1. Tính toán tổng số step cho toàn bộ quá trình Train
epochs = 100
steps_per_epoch = 413  # Thay bằng số step thực tế của bạn (len(img) // batch_size)
total_steps = epochs * steps_per_epoch

# 2. Khởi tạo Cosine Decay Schedule
initial_learning_rate = 0.1 # Khởi đầu lớn giúp SGD bắt nhịp tốt
lr_schedule = CosineDecay(
    initial_learning_rate=initial_learning_rate,
    decay_steps=total_steps,
    alpha=0.001 # Tỷ lệ của LR nhỏ nhất ở step cuối cùng (0.1 * 0.001 = 0.0001)
)

# 3. Khởi tạo SGD VỚI MOMENTUM
# opt = SGD(
#     learning_rate=lr_schedule, 
#     momentum=0.9, 
#     nesterov=True # Nesterov Momentum giúp phanh sớm khi sắp xuống đáy dốc
# )
opt = SGD(
    learning_rate=lr_schedule, 
    momentum=0.9, 
    nesterov=True,
    clipnorm=1.0   # BẮT BUỘC PHẢI CÓ DÒNG NÀY ĐỂ CHỐNG NỔ LOSS
)

# 4. Compile mô hình (Tôi khuyên bạn nên thử thêm Giải pháp Focal Loss ở đây nếu muốn)
model.compile(loss="categorical_crossentropy", optimizer=opt, metrics=['accuracy'])

Model: "MinimNet_MaxAcc"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 32, 32, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 32, 32,    │        288 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        128 │ conv2d_21[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_21       │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_10 │ (None, 32, 32,    │        288 │ activation_21[0]… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        128 │ depthwise_conv2d… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_22       │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ activation_22[0]… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_10          │ (None, 1, 1, 32)  │          0 │ global_average_p… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_21 (Dense)    │ (None, 1, 1, 8)   │        264 │ reshape_10[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_22 (Dense)    │ (None, 1, 1, 32)  │        288 │ dense_21[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_10         │ (None, 32, 32,    │          0 │ activation_22[0]… │
│ (Multiply)          │ 32)               │            │ dense_22[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_22 (Conv2D)  │ (None, 32, 32,    │        512 │ multiply_10[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │         64 │ conv2d_22[0][0]   │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_23 (Conv2D)  │ (None, 32, 32,    │      1,024 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_23[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_23       │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 2,115,774 (8.07 MB)

 Trainable params: 2,098,718 (8.01 MB)

 Non-trainable params: 17,056 (66.62 KB)

Now after we have the model, it is time to start training the model.

## 3. Train the model

Now, we will make an augmentation for the training batch so that it could have more variations of the dataset.<br>
We will also make an additional module from tensorflow such as:

1. EarlyStopping : To stop the training if there's no improvement on the monitored variable,
2. ModelCheckpoint : To save the model/weight if it reached higher value on the monitored variable,
3. ReduceLROnPlateau : To reduce the LR if there's no improvement on the monitored variable.

In [12]:
aug = ImageDataGenerator(rotation_range=10, zoom_range=.05,
                        width_shift_range=.1, height_shift_range=.1,
                        shear_range=.15, horizontal_flip=False,
                        fill_mode='nearest')

eS = EarlyStopping(monitor='val_loss', patience=15, verbose=1, mode='min')
# Lưu file chuẩn định dạng mới (.keras) để tắt cảnh báo h5 legacy
mChk = ModelCheckpoint('OCR_Model.keras', save_best_only=True, monitor='val_loss', mode='min')

In [ ]:
h = model.fit(aug.flow(trainImg, trainLbl, batch_size=batch_size),
             validation_data=(testImg, testLbl),
             steps_per_epoch=len(trainImg)//batch_size,
             epochs=epochs, 
             callbacks=[eS, mChk], 
             class_weight=classWeight, 
             verbose=1)

Epoch 1/100


I0000 00:00:1782677563.775069   78598 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_233282__.460


357/638 ━━━━━━━━━━━━━━━━━━━━ 19s 69ms/step - accuracy: 0.2425 - loss: 6.0353

I0000 00:00:1782677599.724657   78599 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_233282__.460
E0000 00:00:1782677600.178133   78599 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1782677601.117884   78599 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1782677601.490705   84223 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_94', 48 bytes spill stores, 48 bytes spill loads

I0000 00:00:1782677601.501586   78599 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a goo

638/638 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - accuracy: 0.4471 - loss: 4.2027

After training, we will plot the result of the training, such as the loss and accuracy.

Here, we will use the pyplot to plot the result into an image called Result.png

In [ ]:
n = np.array(range(len(h.history['loss'])))
plt.style.use('ggplot')
plt.figure()
plt.plot(n, h.history['loss'], label='Train_loss')
plt.plot(n, h.history['val_loss'], label='Val_loss')
plt.title=('Training Accuracy and Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss/Accuracy')
plt.legend(loc='lower left')
plt.savefig('Result.png')

Then we will test it, and show each class result in F1 score using sklearn classification_report

In [ ]:
predictions = model.predict(testImg, batch_size=batch_size)
print(classification_report(testLbl.argmax(axis=1),
                            predictions.argmax(axis=1), target_names=classLabels))

And its finished! But only for the model part.

Next, we will combine this with OpenCV to make it into real OCR and read multiple character in the images at once.

## 4. Combining with OpenCV
In this step we will:
1. Load an image containing handwriting using OpenCV,
2. Convert the image into Grayscale,
3. Apply some image processing,
4. Get the contours and sort them from left to right,
5. Crop images based on the contours,
6. Predict each image from the cropped image,
7. Append the result to the image.

In [ ]:
# put all the needed variable/parameter
file_name = 'image_to_test.jpg'
image_path = os.path.dirname(notebook_path)+'/'+file_name

image_to_test = cv2.imread(image_path)

#convert to grayscale
gray = cv2.cvtColor(image_to_test, cv2.COLOR_BGR2GRAY)
#blurring the image
blur = cv2.GaussianBlur(gray, (5, 5), 0)

#apply edge filter to the blurred image
edge = cv2.Canny(blur, 30, 150)

In [ ]:
#get contours
cntrs = cv2.findContours(edge.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cntrs = grab_contours(cntrs)

#sort contours from top left to bottom right
boundingBox = [cv2.boundingRect(c) for c in cntrs]
(cntrs, _) = zip(*sorted(zip(cntrs, boundingBox), key=lambda x: (-x[1][1], x[1][0])))

In [ ]:
char_detected = []

for c in cntrs:
    (x,y,w,h) = cv2.boundingRect(c)
    
    #attemp to ignore small contours
    if (w >= 5 and w <= 350) and (h >= 15 and h <= 320):
        #create an Region of Interest
        roi = gray[y:y+h, x:x+w]
        
        #threshold it to make it into binary image
        bin_img = cv2.threshold(roi, 0 ,255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)[1]
        (iH, iW) = bin_img.shape
        
        #if imageWidth(iW) is bigger than the Height, then we will scale the image until
        #the width is equal to 32, else scale the image until we got the height is the 32
        #32 used in here is based on the image_size which is (32,32)
        if iW > iH:
            bin_img = resize(bin_img, width = image_size[0])
        else:
            bin_img = resize(bin_img, height = image_size[1])
            
        #update with the new image size
        (iH, iW) = bin_img.shape
        dX = int(max(0, 32 - iW) / 2.0)
        dY = int(max(0, 32 - iH) / 2.0)
        
        #pad the image and resize it to match the input of the model
        padded = cv2.copyMakeBorder(bin_img, top=dY, bottom=dY,
            left=dX, right=dX, borderType=cv2.BORDER_CONSTANT,
            value=(0, 0, 0))
        padded = cv2.resize(padded, image_size)
        
        #normalize the image so that it got the same value of testImg
        padded = padded.astype("float32") / 255.0
        padded = np.expand_dims(padded, axis=-1)
        
        #put it into list of the queue of images to predict
        char_detected.append([padded, (x, y, w, h)])
        
#get each location and image of detected char
boxes = [b[1] for b in char_detected]
char_detected = np.array([c[0] for c in char_detected], dtype="float32")
#feed it as the input of the model
predictions = model.predict(char_detected)
preds = []
for pred in predictions:
    i = np.argmax(pred)
    prob = pred[i]
    preds.append([classLabels[i], prob])

for (pred, (x, y, w, h)) in zip(preds, boxes):
    #put the result into the image and print it
    print("[DETECTED CHAR] {} - {:.2f}%".format(pred[0], pred[1] * 100))
    cv2.rectangle(image_to_test, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.putText(image_to_test, "{}".format(pred[0]), (x - 10, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)

#show the result
cv2.imwrite("Image_after_test.jpg", image_to_test)

In [ ]:
#show the result
plt.imshow(image_to_test)

As we can see the result shows:
<br>
"This is only a jimple ocr 2020 thij nekds momd troaining"

There are some errors on the S and The insides of R, which always shows up as D or O.<br>
And RE that connected also read as M.

And that's it! Congratulation, you have reached the end of this tutorial!

But as we can see that the result is still not that good.<br>
There are still things that we can do to fix this such as:
1. Make the the minimum size bigger,
2. Maybe train it more or try to use other network,
3. Try to train using other dataset.